In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3, streaming=True) 

In [ ]:
res = llm.invoke("Translate to German: I will call you tomorrow.")
print(res.text)

In [ ]:
from rich.console import Console
from rich.markdown import Markdown

console = Console()
console.print(Markdown(res.text))

In [ ]:
SYSTEM_PROMPT = """
You are a translator for English, Urdu and German.
The user writes in English, Roman Urdu or Urdu script.
Translate the message into casual, everyday German. Use "du", never "Sie".
Reply with the German translation only. No explanations.
Do not translate names.
"""

In [ ]:
FEW_SHOT_EXAMPLES = [
    ("human", "mujhe kal office jana hai"),
    ("ai",    "Ich muss morgen ins Büro."),

    ("human", "yaar mera mood kharab hai 😩"),
    ("ai",    "Alter, meine Laune ist echt mies 😩"),
]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [("system", SYSTEM_PROMPT)]
    + FEW_SHOT_EXAMPLES
    + [
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

In [ ]:
for m in prompt.invoke({"history": [], "input": "kal milte hain"}).to_messages():
    print(m.type, "->", m.content)

In [ ]:
chain = prompt | llm

In [ ]:
res = chain.invoke({"history": [], "input": "main kal tumhein call karunga"})
console.print(Markdown(res.text))

In [ ]:
for chunk in chain.stream({"history": [], "input": "aaj mausam bohot acha hai"}):
    print(chunk.text, end="", flush=True)
print()

In [1]:
from rich.live import Live

text = ""
with Live(console=console, refresh_per_second=8) as live:
    for chunk in chain.stream({"history": [], "input": "aaj mausam bohot acha hai"}):
        text += chunk.text
        live.update(Markdown(text))

NameError: name 'console' is not defined

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

history = []

while True:
    user_text = input("You: ").strip()
    if user_text.lower() in {"exit", "quit"}:
        break
    if not user_text:
        continue

    reply = ""
    with Live(console=console, refresh_per_second=8) as live:
        for chunk in chain.stream({"history": history, "input": user_text}):
            reply += chunk.text
            live.update(Markdown(reply))

    history += [HumanMessage(content=user_text), AIMessage(content=reply)]
    history = history[-10:]